**This notebook aims to ingest the raw csv files and export it to the bronze layer as parquet file. The Bronze layer preserves the raw data structure with minimal transformations (schema enforcement, column normalization, removal of null rows) to ensure traceability and reproducibility. More advanced cleaning and deduplication are deferred to the Silver layer.**

In [1]:
# Import the necessary libraries 
from pyspark.sql import SparkSession
from pathlib import Path
import os
import sqlqueries as sq
import src.enforced_schemas as es
import utils.master as ma
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum as spark_sum, when


#Set the path for logging outputs
job_name = "raw_sales_ingestion"
data_base_path = Path("../Logs") # path for logging data
data_working_path = os.path.join(data_base_path, job_name) 
os.makedirs(data_working_path, exist_ok=True)
ma.set_logging_path(data_working_path)

# Spark initialization locally for development
spark = (
    SparkSession.builder
    .appName("sales-bronze-ingestion")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR") # hide warnings until fixed

ma.log("Spark Session initialized")

# Read raw CSV files with enforced schema
df = (
    spark.read
    .schema(es.schema)
    .option("header", True)
    .csv("../data/raw/Sales_Data")
)
initial_count = df.count()
ma.log(f"Spark DataFrame created from raw CSV files with {initial_count} rows")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/01 17:56:35 WARN Utils: Your hostname, HP-665G11-353, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/01 17:56:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/01 17:56:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-02-01 17:56:44: Spark Session initialized


2026-02-01 17:56:51: Spark DataFrame created from raw CSV files with 186850 rows


**Here we loaded the data from CSV files while applying a predifined schema. The schema is defined in a separate module (enforced_schemas) to keep the notebook clean and reusable. This prevents Spark from inferring incorrect data types and improves read performance. Note that The Order_Date will initially be handled as string and then after investigation casted in order to prevent Spark reading this dimension incorrectly.**

In [2]:
df.printSchema()

root
 |-- Order ID: integer (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity Ordered: integer (nullable = true)
 |-- Price Each: double (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Purchase Address: string (nullable = true)



**Essential Data Cleaning before moving forward**

In [3]:
# Rename column names to remove spaces
df = (df.withColumnRenamed("Order ID", "Order_ID").
      withColumnRenamed("Quantity Ordered", "Quantity_Ordered").
      withColumnRenamed("Price Each", "Price_Each").
      withColumnRenamed("Order Date", "Order_Date").
      withColumnRenamed("Purchase Address", "Purchase_Address")
      )
ma.log("Renamed columns to remove spaces")

# Check the amount of rows where all values are null
null_counts = df.filter(sum(col(c).isNull().cast("int") for c in df.columns) == len(df.columns)).count()
ma.log(f"The total number of null rows is {null_counts}")
    
# Cleaning the amount of rows where all values are null
df = df.dropna(how="all")
final_count = df.count()
ma.log(f"Dropped fully null rows | final_row_count is {final_count}")

#Create temp view to check null row per column 
df.createOrReplaceTempView("sales_data")
ma.log("Temporary view 'sales_data' created from sales df")

ma.log("Check for nulls in each column of sales_data")
spark.sql(sq.nulls_per_column).show()

# Further investigation of nulls - examine the pattern
df_nulls = spark.sql("""
    select * 
    from sales_data
    where Order_ID is null
       or Quantity_Ordered is null
       or Price_Each is null
""").show()

## What happend is that: rows originating from repeated CSV headers were identified by null values in mandatory numeric columns and is plan to be removed prior to downstream transformations ## 

# Filter out rows where headers are mistakenly included in the data
df = df.filter(~(col("Order_Id").isNull()& col("Quantity_Ordered").isNull()& col("Price_Each").isNull()&(col("Product") == "Product")))
df_final_count = df.count()
ma.log(f"Removed header-like rows from raw data | now the rows are {df_final_count}")

2026-02-01 17:56:51: Renamed columns to remove spaces
2026-02-01 17:56:53: The total number of null rows is 545
2026-02-01 17:56:54: Dropped fully null rows | final_row_count is 186305
2026-02-01 17:56:54: Temporary view 'sales_data' created from sales df
2026-02-01 17:56:54: Check for nulls in each column of sales_data
+--------------+-------------+----------------------+----------------+----------------+----------------------+
|Order_ID_nulls|Product_nulls|Quantity_Ordered_nulls|Price_Each_nulls|Order_Date_nulls|Purchase_Address_nulls|
+--------------+-------------+----------------------+----------------+----------------+----------------------+
|           355|            0|                   355|             355|               0|                     0|
+--------------+-------------+----------------------+----------------+----------------+----------------------+

+--------+-------+----------------+----------+----------+----------------+
|Order_ID|Product|Quantity_Ordered|Price_Each|O

**Here the Bronze (raw) layer is as close to the source as possible. Bronze is used for reproducibility and treceability, not performance. Thus, here we dont use partitioning.**

In [4]:
# Write Parquet file to bronze layer 
ma.log("Start ingestion of Parquet file to bronze layer")

(
    df.write
    .mode("overwrite")
    .parquet("../data/bronze/sales")
)
ma.log("Bronze ingestion completed successfully")


2026-02-01 17:56:56: Start ingestion of Parquet file to bronze layer


2026-02-01 17:56:59: Bronze ingestion completed successfully
